Got it ✅ I’ll share the **full code** for the project with MCP server + client for weather, mock pollution MCP, agent factory, and parent agent.

---

# 📂 Final Project Structure

```
order_mgmt_framework/
│── main.py
│
├── config/
│   ├── __init__.py
│   ├── settings.py
│
├── mcp/
│   ├── __init__.py
│   ├── server_weather.py
│   ├── client_weather.py
│   ├── server_pollution.py
│   ├── client_pollution.py
│
├── tools/
│   ├── __init__.py
│   ├── weather_tools.py
│   ├── pollution_tools.py
│
├── agents/
│   ├── __init__.py
│   ├── agent_factory.py
│   ├── parent_agent.py
```

---

# ⚙️ `config/settings.py`

```python
# config/settings.py

AGENT_CONFIG = {
    "weather": {
        "llm": "openai",
        "tools": ["get_city_weather", "get_country_weather"],
        "mcp_servers": ["server_weather"]
    },
    "pollution": {
        "llm": "gemini",
        "tools": ["get_city_pollution", "get_country_pollution"],
        "mcp_servers": ["server_pollution"]
    },
    "parent": {
        "llm": "openai",
        "agents": ["weather", "pollution"],
        "protocol": "A2A",
        "features": ["RAG", "React", "ChainOfThought"]
    }
}
```

---

# 🌦️ `mcp/server_weather.py`

```python
# mcp/server_weather.py
from fastapi import FastAPI
import requests

app = FastAPI()

@app.get("/weather/{location}")
def get_weather(location: str):
    """Return weather info for a given location using wttr.in"""
    try:
        response = requests.get(f"https://wttr.in/{location}?format=3", timeout=5)
        return {"location": location, "weather": response.text}
    except Exception as e:
        return {"error": str(e)}
```

Run:

```bash
uvicorn mcp.server_weather:app --reload --port 8001
```

---

# 📡 `mcp/client_weather.py`

```python
# mcp/client_weather.py
import requests

BASE_URL = "http://127.0.0.1:8001"

def fetch_weather(location: str) -> str:
    """Calls the MCP weather server to fetch weather."""
    try:
        resp = requests.get(f"{BASE_URL}/weather/{location}", timeout=5)
        data = resp.json()
        if "error" in data:
            return f"Error: {data['error']}"
        return data["weather"]
    except Exception as e:
        return f"Client error: {e}"
```

---

# 🏭 `mcp/server_pollution.py` (Mock Server)

```python
# mcp/server_pollution.py
from fastapi import FastAPI

app = FastAPI()

POLLUTION_DATA = {
    "Delhi": "AQI 320 (Very Poor)",
    "Mumbai": "AQI 160 (Moderate)",
    "Paris": "AQI 70 (Good)"
}

@app.get("/pollution/{location}")
def get_pollution(location: str):
    """Mock pollution data"""
    return {"location": location, "pollution": POLLUTION_DATA.get(location, "No data available")}
```

Run:

```bash
uvicorn mcp.server_pollution:app --reload --port 8002
```

---

# 📡 `mcp/client_pollution.py`

```python
# mcp/client_pollution.py
import requests

BASE_URL = "http://127.0.0.1:8002"

def fetch_pollution(location: str) -> str:
    """Calls the MCP pollution server to fetch AQI data."""
    try:
        resp = requests.get(f"{BASE_URL}/pollution/{location}", timeout=5)
        data = resp.json()
        return data.get("pollution", "No data")
    except Exception as e:
        return f"Client error: {e}"
```

---

# 🔧 `tools/weather_tools.py`

```python
# tools/weather_tools.py
from mcp.client_weather import fetch_weather

def get_city_weather(city: str) -> str:
    return fetch_weather(city)

def get_country_weather(country: str) -> str:
    return fetch_weather(country)
```

---

# 🔧 `tools/pollution_tools.py`

```python
# tools/pollution_tools.py
from mcp.client_pollution import fetch_pollution

def get_city_pollution(city: str) -> str:
    return fetch_pollution(city)

def get_country_pollution(country: str) -> str:
    return fetch_pollution(country)
```

---

# 🏭 `agents/agent_factory.py`

```python
# agents/agent_factory.py
from config.settings import AGENT_CONFIG
from tools import weather_tools, pollution_tools

class AgentFactory:
    def __init__(self):
        self.agents = {}

    def build_agent(self, agent_name: str):
        """Dynamically builds an agent from config"""
        if agent_name not in AGENT_CONFIG:
            raise ValueError(f"Unknown agent: {agent_name}")

        config = AGENT_CONFIG[agent_name]
        tool_funcs = []

        # Load tools
        for tool in config.get("tools", []):
            if hasattr(weather_tools, tool):
                tool_funcs.append(getattr(weather_tools, tool))
            elif hasattr(pollution_tools, tool):
                tool_funcs.append(getattr(pollution_tools, tool))

        def run(task: dict):
            city = task.get("city")
            country = task.get("country")
            for tool in tool_funcs:
                if city and "city" in tool.__name__:
                    return tool(city)
                if country and "country" in tool.__name__:
                    return tool(country)
            return f"{agent_name} agent: No location provided"

        self.agents[agent_name] = run
        return run

    def get_agent(self, agent_name: str):
        return self.agents.get(agent_name) or self.build_agent(agent_name)
```

---

# 🧭 `agents/parent_agent.py`

```python
# agents/parent_agent.py

def parent_router(prompt: str) -> str:
    """Simple keyword-based router"""
    if "weather" in prompt.lower():
        return "weather"
    elif "pollution" in prompt.lower() or "aqi" in prompt.lower():
        return "pollution"
    return "end"
```

---

# 🚀 `main.py`

```python
# main.py
from langgraph.graph import StateGraph, START, END
from agents.agent_factory import AgentFactory
from agents.parent_agent import parent_router

def build_graph():
    factory = AgentFactory()
    weather_agent = factory.get_agent("weather")
    pollution_agent = factory.get_agent("pollution")

    builder = StateGraph(dict)

    builder.add_node("parent_router", lambda state: {"next": parent_router(state["prompt"])})
    builder.add_node("weather", lambda state: {"result": weather_agent(state)})
    builder.add_node("pollution", lambda state: {"result": pollution_agent(state)})

    builder.add_edge(START, "parent_router")
    builder.add_conditional_edges(
        "parent_router",
        lambda out: out["next"],
        {"weather": "weather", "pollution": "pollution", "end": END}
    )
    builder.add_edge("weather", END)
    builder.add_edge("pollution", END)

    return builder.compile()

def main():
    graph = build_graph()

    prompts = [
        {"prompt": "What's the weather in Paris?", "city": "Paris"},
        {"prompt": "Check pollution in Delhi", "city": "Delhi"},
    ]

    for p in prompts:
        print(f"\n📝 Prompt: {p['prompt']}")
        result = graph.invoke(p)
        print("🤖 Response:", result.get("result"))

if __name__ == "__main__":
    main()
```

---

# ✅ How to Run

1. Start servers:

   ```bash
   uvicorn mcp.server_weather:app --reload --port 8001
   uvicorn mcp.server_pollution:app --reload --port 8002
   ```

2. Run main app:

   ```bash
   python main.py
   ```

---

👉 This gives you a **LangGraph-based agent framework** with:

* **Weather Agent** (via MCP server → wttr.in)
* **Pollution Agent** (via mock MCP server)
* **Parent Router Agent** (decides which agent to call)

Do you want me to also add **LLM-based routing** (OpenAI/Gemini deciding instead of keyword rules)?
